In [ ]:
METRICS_SRC = 'import warnings\nfrom typing import Literal, NamedTuple\n\nimport polars as pl\nimport tracksdata as td\n\n\nclass EvaluationResult(NamedTuple):\n    """Counts returned by :func:`evaluate`."""\n\n    edge_tp: int\n    edge_fp: int\n    edge_fn: int\n    division_tp: int\n    division_fp: int\n    division_fn: int\n    num_pred_nodes: int\n\n\nclass DatasetsResult(NamedTuple):\n    """Cumulative (micro-averaged) Jaccards plus the combined score."""\n\n    edge_jaccard: float\n    division_jaccard: float\n    score: float\n\n\n# Penalty coefficient for the adjusted edge Jaccard:\n#   J_adj = max(0, J · (1 - ADJUSTMENT_ALPHA · total_node_ratio))\nADJUSTMENT_ALPHA: float = 0.1\n\n# Weight of the division Jaccard in the combined run-level score:\n#   score = adj_edge_jaccard + SCORE_DIVISION_WEIGHT · division_jaccard\nSCORE_DIVISION_WEIGHT: float = 0.1\n\nCOUNT_COLUMNS: tuple[str, ...] = (\n    "edge_tp", "edge_fp", "edge_fn",\n    "division_tp", "division_fp", "division_fn",\n    "num_pred_nodes",\n)\nMETRIC_COLUMNS: tuple[str, ...] = COUNT_COLUMNS + (\n    "node_recall", "total_node_ratio", "edge_jaccard", "adj_edge_jaccard",\n)\n\n\ndef _jaccard(tp: int, fp: int, fn: int) -> float:\n    denom = tp + fp + fn\n    return tp / denom if denom > 0 else float("nan")\n\n\n# function is split for easier testing\ndef _evaluate_matched_graph(\n    graph: td.graph.BaseGraph,\n    gt_graph: td.graph.BaseGraph,\n) -> pl.DataFrame:\n    edge_attrs = graph.edge_attrs(attr_keys=[td.DEFAULT_ATTR_KEYS.MATCHED_EDGE_MASK])\n    # Guard against duplicate edges (same source→target pair appearing multiple times).\n    # tracksdata\'s match() inner-join marks all duplicates as matched, which inflates\n    # the intersection count and can push scores above 1.0. Sort matched rows first\n    # so the dedup keeps the matched copy when duplicates disagree on the mask.\n    edge_attrs = edge_attrs.sort(\n        td.DEFAULT_ATTR_KEYS.MATCHED_EDGE_MASK, descending=True,\n    ).unique(\n        subset=[td.DEFAULT_ATTR_KEYS.EDGE_SOURCE, td.DEFAULT_ATTR_KEYS.EDGE_TARGET],\n        keep="first",\n    )\n    node_attrs = graph.node_attrs(attr_keys=[td.DEFAULT_ATTR_KEYS.NODE_ID, td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID])\n\n    # I\'m assuming valid ground-truth edges are always 100% correct if they have an edge.\n    # Therefore, we don\'t have cases where the cell divided, but not in the ground truth.\n    gt_node_ids = gt_graph.node_ids()\n    gt_node_attrs = pl.DataFrame(\n        {\n            td.DEFAULT_ATTR_KEYS.NODE_ID: gt_node_ids,\n            "out_degree": gt_graph.out_degree(gt_node_ids),\n            "in_degree": gt_graph.in_degree(gt_node_ids),\n        }\n    ).with_columns(\n        (pl.col("out_degree") > 0).alias("out_valid"),\n        (pl.col("in_degree") > 0).alias("in_valid"),\n    )\n\n    # merging ground truth graph into the predicted graph\n    node_attrs = node_attrs.join(\n        gt_node_attrs,\n        left_on=td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID,\n        right_on=td.DEFAULT_ATTR_KEYS.NODE_ID,\n        how="left",\n    ).with_columns(\n        pl.col("out_valid").fill_null(False),\n        pl.col("in_valid").fill_null(False),\n    )\n\n    # merge out valid into source and in valid into target\n    edge_attrs = edge_attrs.join(\n        node_attrs.select(td.DEFAULT_ATTR_KEYS.NODE_ID, "out_valid"),\n        left_on=td.DEFAULT_ATTR_KEYS.EDGE_SOURCE,\n        right_on=td.DEFAULT_ATTR_KEYS.NODE_ID,\n        how="left",\n    ).join(\n        node_attrs.select(td.DEFAULT_ATTR_KEYS.NODE_ID, "in_valid"),\n        left_on=td.DEFAULT_ATTR_KEYS.EDGE_TARGET,\n        right_on=td.DEFAULT_ATTR_KEYS.NODE_ID,\n        how="left",\n    )\n\n    edge_attrs = edge_attrs.with_columns(\n        (pl.col("out_valid") | pl.col("in_valid")).alias("pred_valid"),\n    )\n\n    # sanity check that `pred_valid` is a superset of all matched edges\n    assert edge_attrs.filter(td.DEFAULT_ATTR_KEYS.MATCHED_EDGE_MASK)["pred_valid"].all()\n\n    return edge_attrs\n\n\ndef _compute_score(\n    edge_attrs: pl.DataFrame,\n    gt_num_edges: int,\n    metric: Literal["jaccard", "dice"],\n) -> float:\n    intersection = int(edge_attrs[td.DEFAULT_ATTR_KEYS.MATCHED_EDGE_MASK].sum())\n    n_valid_pred_edges = int(edge_attrs["pred_valid"].sum())\n\n    if metric == "jaccard":\n        num = intersection\n        denom = gt_num_edges + n_valid_pred_edges - intersection\n    elif metric == "dice":\n        num = 2 * intersection\n        denom = gt_num_edges + n_valid_pred_edges\n    else:\n        raise ValueError(f"Invalid metric: {metric}")\n\n    return num / denom if denom > 0 else float("nan")\n\n\ndef _evaluate(\n    graph: td.graph.BaseGraph,\n    gt_graph: td.graph.BaseGraph,\n    metric: Literal["jaccard", "dice"],\n    scale: tuple[float, ...] | None,\n    max_distance: float,\n) -> float:\n    if td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID in graph.node_attr_keys():\n        warnings.warn("Graph already matched, overwriting previous matching.")\n        # Reset matching attributes to defaults before re-matching\n        all_node_ids = graph.node_ids()\n        graph.update_node_attrs(\n            node_ids=all_node_ids,\n            attrs={\n                td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID: -1,\n                td.DEFAULT_ATTR_KEYS.MATCH_SCORE: 0.0,\n            },\n        )\n        all_edge_ids = graph.edge_ids()\n        if len(all_edge_ids) > 0:\n            graph.update_edge_attrs(\n                edge_ids=all_edge_ids,\n                attrs={td.DEFAULT_ATTR_KEYS.MATCHED_EDGE_MASK: False},\n            )\n\n    from tracksdata.metrics import DistanceMatching\n    matching = DistanceMatching(max_distance=max_distance, scale=scale)\n\n    if graph.num_edges() == 0 or graph.num_nodes() == 0:\n        warnings.warn("Predicted graph has no edges or no nodes, returning score 0.0.")\n        return 0.0\n\n    from tracksdata.options import get_options, set_options\n\n    prev_show_progress = get_options().show_progress\n    set_options(show_progress=False)\n    try:\n        with warnings.catch_warnings():\n            from scipy.sparse import SparseEfficiencyWarning\n            warnings.filterwarnings("ignore", category=SparseEfficiencyWarning)\n            graph.match(gt_graph, matching=matching)\n    finally:\n        set_options(show_progress=prev_show_progress)\n\n    edge_attrs = _evaluate_matched_graph(graph, gt_graph)\n\n    return _compute_score(edge_attrs, gt_graph.num_edges(), metric)\n\n\ndef evaluate(\n    graph: td.graph.BaseGraph,\n    gt_graph: td.graph.BaseGraph,\n    scale: tuple[float, ...] | None = None,\n    max_distance: float = 7.0,\n) -> EvaluationResult:\n    """\n    Evaluate a predicted graph against a ground-truth graph using\n    centroid-distance node matching.\n\n    Computes edge TP/FP/FN, division TP/FP/FN (via\n    :func:`biohub_tracking.division_metrics.evaluate_divisions`), and the\n    total number of predicted nodes (irrespective of matching).\n\n    Parameters\n    ----------\n    graph : tracksdata.graph.BaseGraph\n        The predicted graph. Matching attributes are written onto *graph*\n        as a side effect.\n    gt_graph : tracksdata.graph.BaseGraph\n        The ground truth graph.\n    scale : tuple[float, ...] | None, optional\n        Physical scale for each spatial dimension (e.g., (z, y, x)) to\n        account for anisotropy. If None, assumes isotropic data.\n    max_distance : float, optional\n        Maximum distance between centroids to be considered as a match.\n\n    Returns\n    -------\n    EvaluationResult\n    """\n    from .division_metrics import evaluate_divisions\n\n    # Match graph against gt_graph (in place); discard the returned score.\n    _evaluate(graph, gt_graph, "jaccard", scale, max_distance)\n\n    if graph.num_edges() == 0:\n        edge_tp = 0\n        edge_fp = 0\n        edge_fn = gt_graph.num_edges()\n    else:\n        edge_attrs = _evaluate_matched_graph(graph, gt_graph)\n        edge_tp = int(edge_attrs[td.DEFAULT_ATTR_KEYS.MATCHED_EDGE_MASK].sum())\n        edge_valid_pred = int(edge_attrs["pred_valid"].sum())\n        edge_fp = edge_valid_pred - edge_tp\n        edge_fn = gt_graph.num_edges() - edge_tp\n\n    div = evaluate_divisions(\n        graph, gt_graph, scale=scale, max_distance=max_distance,\n    )\n\n    return EvaluationResult(\n        edge_tp=edge_tp,\n        edge_fp=edge_fp,\n        edge_fn=edge_fn,\n        division_tp=div.tp,\n        division_fp=div.fp,\n        division_fn=div.fn,\n        num_pred_nodes=graph.num_nodes(),\n    )\n\n\ndef evaluate_datasets(\n    graph_pairs: list[tuple[td.graph.BaseGraph, td.graph.BaseGraph]],\n    scale: tuple[float, ...] | None = None,\n    max_distance: float = 7.0,\n) -> DatasetsResult:\n    """Run :func:`evaluate` on each (pred, gt) pair and return cumulative\n    (micro-averaged) edge and division Jaccard.\n\n    Per-pair TP/FP/FN counts are summed across the whole list before the\n    Jaccard is computed, so larger datasets dominate the score naturally.\n\n    Parameters\n    ----------\n    graph_pairs : list of (pred_graph, gt_graph)\n        Predicted / ground-truth graph pairs. Each *pred_graph* is mutated\n        in place by matching (same side effect as :func:`evaluate`).\n    scale : tuple[float, ...] | None, optional\n        Physical voxel scale used for centroid-distance matching.\n    max_distance : float, optional\n        Maximum centroid distance for a match.\n\n    Returns\n    -------\n    DatasetsResult\n        Named tuple with ``edge_jaccard``, ``division_jaccard``, and the\n        combined ``score = edge_jaccard + SCORE_DIVISION_WEIGHT *\n        division_jaccard``. If no divisions exist anywhere in the input\n        the division term is dropped and ``score = edge_jaccard``.\n    """\n    edge_tp = edge_fp = edge_fn = 0\n    div_tp = div_fp = div_fn = 0\n    for pred, gt in graph_pairs:\n        r = evaluate(pred, gt, scale=scale, max_distance=max_distance)\n        edge_tp += r.edge_tp\n        edge_fp += r.edge_fp\n        edge_fn += r.edge_fn\n        div_tp += r.division_tp\n        div_fp += r.division_fp\n        div_fn += r.division_fn\n\n    edge_jaccard = _jaccard(edge_tp, edge_fp, edge_fn)\n    has_divisions = (div_tp + div_fp + div_fn) > 0\n    division_jaccard = _jaccard(div_tp, div_fp, div_fn) if has_divisions else float("nan")\n    score = edge_jaccard + SCORE_DIVISION_WEIGHT * division_jaccard if has_divisions else edge_jaccard\n\n    return DatasetsResult(\n        edge_jaccard=edge_jaccard,\n        division_jaccard=division_jaccard,\n        score=score,\n    )\n\n\ndef _matched_node_ids(graph: td.graph.BaseGraph) -> pl.DataFrame:\n    """Return a DataFrame with NODE_ID and MATCHED_NODE_ID (as Int64) for *graph*."""\n    node_attrs = graph.node_attrs(\n        attr_keys=[td.DEFAULT_ATTR_KEYS.NODE_ID, td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID]\n    )\n    return node_attrs\n\n\ndef node_recall(\n    graph: td.graph.BaseGraph,\n    gt_graph: td.graph.BaseGraph,\n) -> float:\n    """Fraction of GT nodes that were matched by a predicted node.\n\n    The predicted graph must already be matched (e.g. via :func:`evaluate` or\n    ``graph.match``).\n    """\n    node_attrs = _matched_node_ids(graph)\n    matched = node_attrs.filter(\n        pl.col(td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID).is_not_null()\n        & (pl.col(td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID) != -1)\n    )\n    n_matched_gt = matched[td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID].n_unique()\n    return n_matched_gt / gt_graph.num_nodes()\n\n\ndef per_sample_metrics(\n    er: EvaluationResult,\n    n_total: float,\n    node_recall: float,\n) -> dict:\n    """Derive per-sample metric columns from an :class:`EvaluationResult`.\n\n    Computes ``edge_jaccard``, ``total_node_ratio`` (``(N_pred − N_total) / N_total``),\n    and the adjusted edge Jaccard ``J_adj = max(0, J · (1 − α · total_node_ratio))``\n    with α = :data:`ADJUSTMENT_ALPHA`.\n\n    Parameters\n    ----------\n    er\n        Counts for one (pred, gt) pair — see :func:`evaluate`.\n    n_total\n        Target node count (e.g. from the GEFF ``estimated_number_of_nodes``\n        metadata extra). Pass ``float("nan")`` when unavailable; that makes\n        ``total_node_ratio`` and ``adj_edge_jaccard`` also NaN.\n    node_recall\n        Fraction of GT nodes matched by a predicted node.\n\n    Returns\n    -------\n    dict\n        One entry per key in :data:`METRIC_COLUMNS`.\n    """\n    if n_total > 0:\n        total_node_ratio = (er.num_pred_nodes - n_total) / n_total\n    else:\n        total_node_ratio = float("nan")\n\n    edge_denom = er.edge_tp + er.edge_fp + er.edge_fn\n    edge_jaccard = er.edge_tp / edge_denom if edge_denom > 0 else float("nan")\n    if edge_jaccard == edge_jaccard and total_node_ratio == total_node_ratio:\n        adj_edge_jaccard = max(\n            0.0, edge_jaccard * (1 - ADJUSTMENT_ALPHA * total_node_ratio),\n        )\n    else:\n        adj_edge_jaccard = float("nan")\n\n    return {\n        "edge_tp": er.edge_tp, "edge_fp": er.edge_fp, "edge_fn": er.edge_fn,\n        "division_tp": er.division_tp,\n        "division_fp": er.division_fp,\n        "division_fn": er.division_fn,\n        "num_pred_nodes": er.num_pred_nodes,\n        "node_recall": node_recall,\n        "total_node_ratio": total_node_ratio,\n        "edge_jaccard": edge_jaccard,\n        "adj_edge_jaccard": adj_edge_jaccard,\n    }\n\n\ndef nan_metrics_row() -> dict:\n    """Return a dict with every :data:`METRIC_COLUMNS` key set to NaN."""\n    return {col: float("nan") for col in METRIC_COLUMNS}\n\n\ndef summarise(rows: list[dict]) -> dict:\n    """Aggregate per-sample metric rows into a run-level summary.\n\n    - ``edge_jaccard`` / ``division_jaccard``: micro-averaged across valid rows\n      (TP/FP/FN summed, then Jaccard).\n    - ``adj_edge_jaccard``: per-sample adjusted Jaccard weight-averaged by\n      sample size ``w_i = TP_i + FP_i + FN_i``; rows with NaN are skipped.\n    - ``score``: ``adj_edge_jaccard + SCORE_DIVISION_WEIGHT · division_jaccard``.\n\n    Parameters\n    ----------\n    rows\n        Per-sample dicts as produced by :func:`per_sample_metrics`. Rows with\n        NaN ``edge_tp`` are treated as failed evaluations and skipped.\n    """\n    valid = [r for r in rows if r["edge_tp"] == r["edge_tp"]]\n    if not valid:\n        return {\n            "n": 0, "edge_jaccard": float("nan"),\n            "division_jaccard": float("nan"),\n            "division_tp": 0, "division_fp": 0, "division_fn": 0,\n            "node_recall": float("nan"),\n            "adj_edge_jaccard": float("nan"), "n_adj": 0,\n            "score": float("nan"),\n        }\n    totals = {c: sum(r[c] for r in valid) for c in COUNT_COLUMNS}\n\n    adj_rows = [r for r in valid if r["adj_edge_jaccard"] == r["adj_edge_jaccard"]]\n    weights = [r["edge_tp"] + r["edge_fp"] + r["edge_fn"] for r in adj_rows]\n    total_w = sum(weights)\n    if total_w > 0:\n        adj_edge_jaccard = sum(\n            w * r["adj_edge_jaccard"] for w, r in zip(weights, adj_rows)\n        ) / total_w\n    else:\n        adj_edge_jaccard = float("nan")\n\n    division_total = (\n        totals["division_tp"] + totals["division_fp"] + totals["division_fn"]\n    )\n    if division_total == 0:\n        warnings.warn(\n            "No divisions present across any sample in this split; "\n            "dropping division term from the combined score."\n        )\n        division_jaccard = float("nan")\n        score = adj_edge_jaccard\n    else:\n        division_jaccard = _jaccard(\n            totals["division_tp"], totals["division_fp"], totals["division_fn"],\n        )\n        score = adj_edge_jaccard + SCORE_DIVISION_WEIGHT * division_jaccard\n    return {\n        "n": len(valid),\n        "edge_jaccard": _jaccard(\n            totals["edge_tp"], totals["edge_fp"], totals["edge_fn"],\n        ),\n        "division_jaccard": division_jaccard,\n        "division_tp": totals["division_tp"],\n        "division_fp": totals["division_fp"],\n        "division_fn": totals["division_fn"],\n        "node_recall": sum(r["node_recall"] for r in valid) / len(valid),\n        "adj_edge_jaccard": adj_edge_jaccard,\n        "n_adj": len(adj_rows),\n        "score": score,\n    }\n'
DIVISION_SRC = 'import warnings\nfrom collections import deque\nfrom typing import NamedTuple\n\nimport polars as pl\nimport tracksdata as td\n\n\nclass DivisionCounts(NamedTuple):\n    """Counts for division event evaluation."""\n\n    tp: int\n    fn: int\n    fp: int\n\n\ndef _reset_matching_attrs(graph: td.graph.BaseGraph) -> None:\n    """Reset any pre-existing match attrs in place so a fresh ``.match()`` isn\'t\n    contaminated by stale values carried in from a previous matching pass."""\n    node_keys = graph.node_attr_keys()\n    if td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID in node_keys:\n        node_ids = graph.node_ids()\n        if len(node_ids) > 0:\n            reset: dict = {td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID: -1}\n            if td.DEFAULT_ATTR_KEYS.MATCH_SCORE in node_keys:\n                reset[td.DEFAULT_ATTR_KEYS.MATCH_SCORE] = 0.0\n            graph.update_node_attrs(node_ids=node_ids, attrs=reset)\n    if td.DEFAULT_ATTR_KEYS.MATCHED_EDGE_MASK in graph.edge_attr_keys():\n        edge_ids = graph.edge_ids()\n        if len(edge_ids) > 0:\n            graph.update_edge_attrs(\n                edge_ids=edge_ids,\n                attrs={td.DEFAULT_ATTR_KEYS.MATCHED_EDGE_MASK: False},\n            )\n\n\ndef extract_divisions(\n    graph: td.graph.BaseGraph,\n) -> dict[int, td.graph.BaseGraph]:\n    """Extract individual division events as separate subgraphs.\n\n    Each division event includes the parent of the dividing node, the\n    dividing node, its children, and the grandchildren::\n\n        parent → divider → child1 → grandchild1\n                         → child2 → grandchild2\n\n    Parameters\n    ----------\n    graph : td.graph.BaseGraph\n        The input tracking graph.\n\n    Returns\n    -------\n    dict[int, td.graph.BaseGraph]\n        Mapping from dividing node ID to a subgraph containing the\n        parent, divider, children, and grandchildren.\n    """\n    divisions: dict[int, td.graph.BaseGraph] = {}\n    for div_node in graph.dividing_nodes():\n        parents = graph.predecessors(div_node)\n        children = graph.successors(div_node)\n        grandchildren = [gc for child in children for gc in graph.successors(child)]\n        keep = [*parents, div_node, *children, *grandchildren]\n        divisions[div_node] = graph.filter(node_ids=keep).subgraph()\n    return divisions\n\n\ndef match_divisions(\n    pred_graph: td.graph.BaseGraph,\n    gt_graph: td.graph.BaseGraph,\n    scale: tuple[float, ...] | None = None,\n    max_distance: float = 7.0,\n) -> dict[int, td.graph.BaseGraph]:\n    """Match the predicted graph against each GT division subgraph.\n\n    Extracts division events from *gt_graph* via :func:`extract_divisions`,\n    then runs ``pred_graph.match(gt_div, ...)`` for each one independently.\n    A fresh copy of *pred_graph* is used per division so matchings don\'t\n    interfere.\n\n    Parameters\n    ----------\n    pred_graph : td.graph.BaseGraph\n        The predicted tracking graph.\n    gt_graph : td.graph.BaseGraph\n        The ground-truth tracking graph.\n    scale : tuple[float, ...] | None\n        Physical voxel scale used for centroid-distance matching.\n    max_distance : float\n        Maximum centroid distance for a match.\n\n    Returns\n    -------\n    dict[int, td.graph.BaseGraph]\n        Mapping from GT dividing-node ID to the matched copy of\n        *pred_graph* for that division.\n    """\n    from tracksdata.metrics import DistanceMatching\n    matching = DistanceMatching(max_distance=max_distance, scale=scale)\n\n    gt_divisions = extract_divisions(gt_graph)\n    matched: dict[int, td.graph.BaseGraph] = {}\n\n    from tracksdata.options import get_options, set_options\n\n    prev_show_progress = get_options().show_progress\n    set_options(show_progress=False)\n    try:\n        for div_node, gt_div in gt_divisions.items():\n            pred_copy = pred_graph.copy()\n            _reset_matching_attrs(pred_copy)\n            with warnings.catch_warnings():\n                from scipy.sparse import SparseEfficiencyWarning\n                warnings.filterwarnings("ignore", category=SparseEfficiencyWarning)\n                pred_copy.match(gt_div, matching=matching)\n            matched[div_node] = pred_copy\n    finally:\n        set_options(show_progress=prev_show_progress)\n\n    return matched\n\n\ndef _match_full(\n    pred_graph: td.graph.BaseGraph,\n    gt_graph: td.graph.BaseGraph,\n    scale: tuple[float, ...] | None,\n    max_distance: float,\n) -> td.graph.BaseGraph:\n    """Match the full pred graph against the full GT graph, return the matched copy."""\n    from tracksdata.metrics import DistanceMatching\n    matching = DistanceMatching(max_distance=max_distance, scale=scale)\n\n    pred_copy = pred_graph.copy()\n    _reset_matching_attrs(pred_copy)\n\n    from tracksdata.options import get_options, set_options\n\n    prev_show_progress = get_options().show_progress\n    set_options(show_progress=False)\n    try:\n        with warnings.catch_warnings():\n            from scipy.sparse import SparseEfficiencyWarning\n            warnings.filterwarnings("ignore", category=SparseEfficiencyWarning)\n            pred_copy.match(gt_graph, matching=matching)\n    finally:\n        set_options(show_progress=prev_show_progress)\n\n    return pred_copy\n\n\ndef _matched_node_attrs(graph: td.graph.BaseGraph) -> pl.DataFrame:\n    """Return node attrs (node_id, matched_node_id, t) for matched pred nodes."""\n    node_attrs = graph.node_attrs(\n        attr_keys=[\n            td.DEFAULT_ATTR_KEYS.NODE_ID,\n            td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID,\n            "t",\n        ],\n    )\n    return node_attrs.filter(\n        pl.col(td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID).is_not_null()\n        & (pl.col(td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID) != -1)\n    )\n\n\ndef _has_stage_coverage(\n    matched_attrs: pl.DataFrame,\n    gt_div: td.graph.BaseGraph,\n    divider_id: int,\n) -> bool:\n    """Check that matches cover both stages of a GT division.\n\n    The GT division subgraph has a *one-node stage* (timepoints with a\n    single GT node — parent and divider, pre-split) and two or more\n    *daughter lineages* (each child of *divider_id* plus its descendants\n    within the subgraph). A valid match requires:\n\n    * ≥1 matched prediction node whose timepoint falls in the one-node\n      stage, AND\n    * matched prediction nodes whose matched GT nodes cover ≥2 distinct\n      daughter lineages. Lineage hits may occur at different timepoints;\n      a single daughter matched only at t=divider+2 still counts.\n\n    When the subgraph contains a secondary divider (e.g. successive\n    divisions), *divider_id* disambiguates which split we\'re scoring.\n    """\n    if matched_attrs.is_empty():\n        return False\n\n    gt_time_counts = (\n        gt_div.node_attrs(attr_keys=["t"])\n        .group_by("t")\n        .agg(pl.len().alias("n"))\n    )\n    one_node_times = set(gt_time_counts.filter(pl.col("n") == 1)["t"].to_list())\n    if not one_node_times:\n        return False\n\n    children = gt_div.successors(divider_id)\n    if len(children) < 2:\n        return False\n\n    def _descendants(seed: int) -> set[int]:\n        out: set[int] = {seed}\n        stack = [seed]\n        while stack:\n            for nxt in gt_div.successors(stack.pop()):\n                if nxt not in out:\n                    out.add(nxt)\n                    stack.append(nxt)\n        return out\n\n    lineages = [_descendants(c) for c in children]\n\n    matched_time_counts = matched_attrs.group_by("t").agg(pl.len().alias("n"))\n    has_one = matched_time_counts.filter(pl.col("t").is_in(one_node_times)).height > 0\n    if not has_one:\n        return False\n\n    matched_gt_ids = set(matched_attrs[td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID].to_list())\n    lineages_covered = sum(1 for lin in lineages if lin & matched_gt_ids)\n    return lineages_covered >= 2\n\n\ndef _weakly_connected_components(\n    graph: td.graph.BaseGraph,\n    node_ids: list[int],\n) -> list[tuple[set[int], set[int]]]:\n    """Partition *node_ids* into weakly-connected components of *graph*.\n\n    Returns one ``(matched_subset, visited)`` pair per component:\n    *matched_subset* is the component restricted to *node_ids*; *visited*\n    is every graph node reachable from the component (including unmatched\n    intermediaries). The visited set lets callers locate structural\n    features -- in particular pred dividing nodes -- that may sit on\n    unmatched nodes between matched ones.\n    """\n    remaining = set(node_ids)\n    components: list[tuple[set[int], set[int]]] = []\n    while remaining:\n        seed = next(iter(remaining))\n        visited: set[int] = {seed}\n        queue: deque[int] = deque([seed])\n        component: set[int] = {seed}\n        while queue:\n            current = queue.popleft()\n            neighbors = graph.successors(current) + graph.predecessors(current)\n            for neighbor in neighbors:\n                if neighbor not in visited:\n                    visited.add(neighbor)\n                    queue.append(neighbor)\n                    if neighbor in remaining:\n                        component.add(neighbor)\n        components.append((component, visited))\n        remaining -= component\n    return components\n\n\ndef _bipartite_max_matching(\n    left: list[int],\n    edges: dict[int, set[int]],\n) -> dict[int, int]:\n    """Maximum-cardinality bipartite matching via DFS augmenting paths.\n\n    *edges* maps each left-side vertex to the set of adjacent right-side\n    vertices. Returns only the matched pairs as a ``left → right`` dict.\n    """\n    match_r: dict[int, int] = {}\n    match_l: dict[int, int] = {}\n\n    def augment(u: int, seen: set[int]) -> bool:\n        for v in edges.get(u, ()):\n            if v in seen:\n                continue\n            seen.add(v)\n            if v not in match_r or augment(match_r[v], seen):\n                match_l[u] = v\n                match_r[v] = u\n                return True\n        return False\n\n    for u in left:\n        augment(u, set())\n\n    return match_l\n\n\ndef score_divisions(\n    pred_graph: td.graph.BaseGraph,\n    gt_graph: td.graph.BaseGraph,\n    scale: tuple[float, ...] | None = None,\n    max_distance: float = 7.0,\n) -> dict[int, int]:\n    """Score each GT division: 1 if the prediction recovers it, 0 otherwise.\n\n    For each GT division, the predicted graph is matched against the\n    division subgraph and checked for a spanning component satisfying:\n\n    1. At least one matched prediction node in the GT\'s one-node stage\n       (pre-division timepoints).\n    2. At least two matched prediction nodes at the same timepoint in the\n       GT\'s two-node stage (post-division timepoints).\n    3. All matched prediction nodes in a single weakly-connected\n       component of the prediction graph.\n\n    Each such spanning component is associated with the *pred dividing\n    nodes* (out-degree ≥ 2) it contains. A maximum-cardinality bipartite\n    matching is then computed so each pred dividing node serves at most\n    one GT division, and each GT division is paired with at most one\n    pred dividing node. A GT division scores 1 only if it is paired in\n    that matching -- this prevents a single pred fork from being\n    credited to multiple GT divisions.\n\n    Parameters\n    ----------\n    pred_graph : td.graph.BaseGraph\n        The predicted tracking graph.\n    gt_graph : td.graph.BaseGraph\n        The ground-truth tracking graph.\n    scale : tuple[float, ...] | None\n        Physical voxel scale used for centroid-distance matching.\n    max_distance : float\n        Maximum centroid distance for a match.\n\n    Returns\n    -------\n    dict[int, int]\n        Mapping from GT dividing-node ID to 1 (paired) or 0 (not).\n    """\n    matched = match_divisions(\n        pred_graph, gt_graph, scale, max_distance,\n    )\n    gt_divisions = extract_divisions(gt_graph)\n    pred_div_nodes = set(pred_graph.dividing_nodes())\n\n    candidates: dict[int, set[int]] = {}\n    for div_node, matched_pred in matched.items():\n        matched_attrs = _matched_node_attrs(matched_pred)\n        node_ids = matched_attrs[td.DEFAULT_ATTR_KEYS.NODE_ID].to_list()\n        components = _weakly_connected_components(matched_pred, node_ids)\n        gt_div = gt_divisions[div_node]\n        div_candidates: set[int] = set()\n        for matched_subset, visited in components:\n            comp_attrs = matched_attrs.filter(\n                pl.col(td.DEFAULT_ATTR_KEYS.NODE_ID).is_in(list(matched_subset))\n            )\n            if _has_stage_coverage(comp_attrs, gt_div, div_node):\n                div_candidates |= visited & pred_div_nodes\n        candidates[div_node] = div_candidates\n\n    pairing = _bipartite_max_matching(list(candidates), candidates)\n    return {div: int(div in pairing) for div in candidates}\n\n\ndef count_matched_pred_divisions(\n    pred_graph: td.graph.BaseGraph,\n    gt_graph: td.graph.BaseGraph,\n    scale: tuple[float, ...] | None = None,\n    max_distance: float = 7.0,\n) -> int:\n    """Count predicted division nodes whose matched GT node is annotated.\n\n    Matches the full predicted graph against the full GT graph.  Among\n    predicted nodes that were matched to a GT node, counts how many are\n    dividing (out-degree >= 2) in the prediction *and* whose matched GT\n    node has at least one child.  A matched GT node with no children marks\n    the end of the annotation — we can\'t tell whether the cell actually\n    divided there, so such predicted divisions are excluded from the count\n    (and therefore from the FP tally).\n\n    Parameters\n    ----------\n    pred_graph : td.graph.BaseGraph\n        The predicted tracking graph.\n    gt_graph : td.graph.BaseGraph\n        The ground-truth tracking graph.\n    scale : tuple[float, ...] | None\n        Physical voxel scale used for centroid-distance matching.\n    max_distance : float\n        Maximum centroid distance for a match.\n\n    Returns\n    -------\n    int\n        Number of matched predicted division nodes.\n    """\n    matched_pred = _match_full(\n        pred_graph, gt_graph, scale, max_distance,\n    )\n\n    node_attrs = matched_pred.node_attrs(\n        attr_keys=[td.DEFAULT_ATTR_KEYS.NODE_ID, td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID],\n    )\n    matched_nodes = node_attrs.filter(\n        pl.col(td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID).is_not_null()\n        & (pl.col(td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID) != -1)\n    )\n\n    count = 0\n    for row in matched_nodes.iter_rows(named=True):\n        pred_node = row[td.DEFAULT_ATTR_KEYS.NODE_ID]\n        gt_node = row[td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID]\n        if (\n            matched_pred.out_degree(pred_node) >= 2\n            and gt_graph.out_degree(gt_node) >= 1\n        ):\n            count += 1\n    return count\n\n\ndef evaluate_divisions(\n    pred_graph: td.graph.BaseGraph,\n    gt_graph: td.graph.BaseGraph,\n    scale: tuple[float, ...] | None = None,\n    max_distance: float = 7.0,\n) -> DivisionCounts:\n    """Compute TP, FN, and FP counts for division events.\n\n    - **TP**: GT divisions correctly recovered in the prediction\n      (matched nodes connected and forking).\n    - **FN**: GT divisions not recovered.\n    - **FP**: Predicted divisions whose matched GT node is not dividing.\n\n    Parameters\n    ----------\n    pred_graph : td.graph.BaseGraph\n        The predicted tracking graph.\n    gt_graph : td.graph.BaseGraph\n        The ground-truth tracking graph.\n    scale : tuple[float, ...] | None\n        Physical voxel scale used for centroid-distance matching.\n    max_distance : float\n        Maximum centroid distance for a match.\n\n    Returns\n    -------\n    DivisionCounts\n        Named tuple with ``tp``, ``fn``, and ``fp`` fields.\n    """\n    scores = score_divisions(\n        pred_graph, gt_graph, scale, max_distance,\n    )\n    tp = sum(scores.values())\n    fn = len(scores) - tp\n    matched_pred_divs = count_matched_pred_divisions(\n        pred_graph, gt_graph, scale, max_distance,\n    )\n    fp = max(0, matched_pred_divs - tp)\n    return DivisionCounts(tp=tp, fn=fn, fp=fp)\n'
SUBMISSIONS = {'ttasec': '/kaggle/input/notebooks/vigneshnehru/claude-eval-ttasec/submission.csv', 'ttaz16': '/kaggle/input/notebooks/vigneshnehru/claude-eval-ttaz16/submission.csv'}
IMPL_SRC = '"""Scoring, run in its own interpreter so the freshly installed polars loads clean."""\nimport json, sys, traceback\nfrom pathlib import Path\n\nimport pandas as pd\nimport polars as pl\nimport tracksdata as td\n\nsys.path.insert(0, "/kaggle/working")\nfrom bt import metrics as M\n\nprint("polars", pl.__version__, "| tracksdata ok", flush=True)\n\nCOMP = Path("/kaggle/input/competitions/biohub-cell-tracking-during-development")\nif not COMP.exists():\n    COMP = Path("/kaggle/input/biohub-cell-tracking-during-development")\nTRAIN = COMP / "train"\nVOXEL = (1.625, 0.40625, 0.40625)\n\n\ndef graph_from_geff(path):\n    """Read the GT graph WITHOUT tracksdata\'s own geff loader.\n\n    `IndexedRXGraph.from_geff` calls `pl.Series([value])` on each default attribute, and on\n    these ground-truth files that lands in a polars branch referencing `PySeries`, which this\n    polars only imports under TYPE_CHECKING:\n\n        File "polars/_utils/construction/series.py", line 322, in sequence_to_pyseries\n            elif python_dtype == PySeries:\n        NameError: name \'PySeries\' is not defined\n\n    The arm notebooks never hit it because they only ever load *prediction* geffs they wrote\n    themselves. Reading via `geff` into networkx and then building the graph with the same\n    code that builds the prediction graph sidesteps the incompatibility -- and has the better\n    property that both sides of the comparison are now constructed identically.\n    """\n    import geff\n    nxg, _meta = geff.read(str(path), backend="networkx")\n    return graph_from_nodes(\n        [(n, d) for n, d in nxg.nodes(data=True)],\n        list(nxg.edges()),\n        label=str(path.name),\n    )\n\n\ndef graph_from_nodes(nodes, edges, label=""):\n    g = td.graph.IndexedRXGraph()\n    for k in ("z", "y", "x"):\n        g.add_node_attr_key(k, 0.0)\n    idx = {}\n    for nid, d in nodes:\n        missing = [k for k in ("t", "z", "y", "x") if k not in d]\n        if missing:\n            raise KeyError(f"{label}: node {nid} lacks {missing}; has {sorted(d)}")\n        idx[nid] = g.add_node({"t": int(d["t"]), "z": float(d["z"]),\n                               "y": float(d["y"]), "x": float(d["x"])})\n    for u, v in edges:\n        g.add_edge(source_id=idx[u], target_id=idx[v], attrs={})\n    return g\n\n\ndef node_budget(geff_path):\n    try:\n        from geff import GeffMetadata\n        meta = GeffMetadata.read(str(geff_path))\n        v = (meta.extra or {}).get("estimated_number_of_nodes")\n        return float(v) if v is not None else float("nan")\n    except Exception as e:\n        print("   node budget unavailable:", e, flush=True)\n        return float("nan")\n\n\ndef graph_from_rows(nodes, edges):\n    """Build a tracksdata graph from submission rows.\n\n    Every attribute key must be declared before the first `add_node`; a fresh graph knows\n    only `t`, which is what killed the first attempt. Verified locally against a real\n    submission: 25,622 nodes in 0.6s.\n    """\n    return graph_from_nodes(\n        [(int(r.node_id), {"t": r.t, "z": r.z, "y": r.y, "x": r.x})\n         for r in nodes.itertuples(index=False)],\n        [(int(r.source_id), int(r.target_id)) for r in edges.itertuples(index=False)],\n        label="submission")\n\n\nresults = {}\nfor name, csv_path in json.loads(sys.argv[1]).items():\n    print("=" * 70, flush=True)\n    print(name, csv_path, flush=True)\n    if not Path(csv_path).exists():\n        print("   MISSING -- mount is not where expected", flush=True)\n        for p in sorted(Path("/kaggle/input").glob("*/*/*")):\n            print("     ", p, flush=True)\n        continue\n    sub = pd.read_csv(csv_path)\n    rows = []\n    for ds in sorted(sub["dataset"].astype(str).unique()):\n        d = sub[sub["dataset"].astype(str) == ds]\n        gt_path = TRAIN / f"{ds}.geff"\n        if not gt_path.exists():\n            print(f"   {ds}: no ground truth at {gt_path}", flush=True)\n            continue\n        try:\n            pred = graph_from_rows(d[d["row_type"] == "node"], d[d["row_type"] == "edge"])\n            gt = graph_from_geff(gt_path)\n            er = M.evaluate(pred, gt, VOXEL)\n            row = M.per_sample_metrics(er, node_budget(gt_path), M.node_recall(pred, gt))\n            row["dataset"] = ds\n            rows.append(row)\n            print(f"   {ds:<16} edge_J={row[\'edge_jaccard\']:.4f} "\n                  f"adj={row[\'adj_edge_jaccard\']:.4f} "\n                  f"div tp/fp/fn={row[\'division_tp\']}/{row[\'division_fp\']}/{row[\'division_fn\']} "\n                  f"ratio={row[\'total_node_ratio\']:+.4f}", flush=True)\n        except Exception:\n            print(f"   {ds}: FAILED", flush=True)\n            traceback.print_exc()\n    if rows:\n        s = M.summarise(rows)\n        results[name] = {"summary": s, "per_dataset": rows}\n        print(f"\\n   SUMMARY {name}", flush=True)\n        for k in ("n", "edge_jaccard", "adj_edge_jaccard", "division_jaccard",\n                  "division_tp", "division_fp", "division_fn", "node_recall", "score"):\n            if k in s:\n                print(f"      {k:<20} {s[k]}", flush=True)\n\nprint("\\n" + "=" * 70, flush=True)\nfor name, r in results.items():\n    print(f"FINAL {name:<24} score={r[\'summary\'].get(\'score\')} "\n          f"adj_edge={r[\'summary\'].get(\'adj_edge_jaccard\')} "\n          f"div_J={r[\'summary\'].get(\'division_jaccard\')}", flush=True)\nPath("/kaggle/working/score_summary.json").write_text(json.dumps(results, indent=1, default=str))\n'
import json, subprocess, sys
from pathlib import Path

# Install from the support pack's OFFLINE WHEELS, with --no-deps, which is what every arm
# notebook does and what its own log explains: "Dependency resolver is disabled with
# --no-deps to avoid replacing Kaggle numpy/scipy in a live kernel."
_wheels = next((p.parent for p in Path("/kaggle/input").glob("*/**/tracksdata-*.whl")), None)
if _wheels is None:
    _wheels = next((p for p in Path("/kaggle/input").glob("*/**/wheels") if p.is_dir()), None)
print("wheel dir:", _wheels, flush=True)
if _wheels is None:
    for _p in sorted(Path("/kaggle/input").glob("*/*")):
        print("   mounted:", _p, flush=True)
    raise RuntimeError("no offline wheels mounted -- add the support pack as a data source")

# polars needs --force-reinstall: the image ships an older one, pip calls the requirement
# satisfied and skips it, and tracksdata then raises `no attribute 'Float16'`.
for _stage, _pkgs, _force in (
        ("polars", ["polars"], True),
        # The arm notebooks' list, verbatim. tracksdata imports its solvers at package load,
        # so ilpy and pyscipopt are needed whether or not anything is solved here.
        ("graph stack", ["tracksdata", "zarr", "pyscipopt", "geff", "geff_spec", "ilpy",
                         "imagecodecs", "rustworkx", "numcodecs", "donfig", "bidict"],
         False)):
    _cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps",
            "--find-links", str(_wheels)] + (["--force-reinstall"] if _force else []) + _pkgs
    _r = subprocess.run(_cmd, capture_output=True, text=True)
    print(f"pip [{_stage}] rc {_r.returncode}", flush=True)
    if _r.returncode:
        print(_r.stdout[-1200:], _r.stderr[-1200:], flush=True)

# The official metric code, carried inline -- the same bytes the support pack ships at
# src/biohub_tracking/{metrics,division_metrics}.py. `metrics` does a relative
# `from .division_metrics import evaluate_divisions`, so both live in one package.
_pkg = Path("/kaggle/working/bt")
_pkg.mkdir(parents=True, exist_ok=True)
(_pkg / "__init__.py").write_text("")
(_pkg / "metrics.py").write_text(METRICS_SRC)
(_pkg / "division_metrics.py").write_text(DIVISION_SRC)
Path("/kaggle/working/score_impl.py").write_text(IMPL_SRC)

# Run the scoring in a FRESH INTERPRETER. Force-reinstalling polars under a process that has
# already imported it leaves a half-replaced package: the version string comes back empty and
# every dataset dies on `NameError: name 'PySeries' is not defined`. The arm notebooks never
# see this because they install in the notebook and predict in a subprocess; so does this.
_rc = subprocess.run([sys.executable, "/kaggle/working/score_impl.py",
                      json.dumps(SUBMISSIONS)], cwd="/kaggle/working")
print("scoring rc", _rc.returncode, flush=True)
if _rc.returncode:
    raise RuntimeError(f"scoring failed rc={_rc.returncode}")
